In [1]:
!pip install krippendorff pingouin

In [3]:
import re
import numpy as np
import pandas as pd

In [4]:
csv_path = "/home/infai/Desktop/UI_SysBio_Dec03_2025/UI_SYSBIO_Sol_Lib/RUN_DEC_16/llm_judge_report_evaluation/outputs/bioasq/direct/report_n_references/files/llm_evaluation_without_averages.csv"  # <-- change
df = pd.read_csv(csv_path)
df.columns

Index(['id', 'wtnp_openai/gpt-oss-120b_readability',
       'wtp_openai/gpt-oss-120b_readability',
       'wot_openai/gpt-oss-120b_readability',
       'wot_openai/gpt-oss-120b_substance_density',
       'wtnp_openai/gpt-oss-120b_substance_density',
       'wtp_openai/gpt-oss-120b_substance_density',
       'wtp_openai/gpt-oss-120b_relevance',
       'wtnp_openai/gpt-oss-120b_relevance',
       'wot_openai/gpt-oss-120b_relevance',
       'wtnp_openai/gpt-oss-120b_expert_suitability',
       'wtp_openai/gpt-oss-120b_expert_suitability',
       'wot_openai/gpt-oss-120b_expert_suitability',
       'wot_openai/gpt-oss-120b_coverage', 'wtp_openai/gpt-oss-120b_coverage',
       'wtnp_openai/gpt-oss-120b_coverage',
       'wtnp_Qwen/Qwen3-Coder-30B-A3B-Instruct_readability',
       'wtp_Qwen/Qwen3-Coder-30B-A3B-Instruct_readability',
       'wot_Qwen/Qwen3-Coder-30B-A3B-Instruct_readability',
       'wot_Qwen/Qwen3-Coder-30B-A3B-Instruct_substance_density',
       'wtp_Qwen/Qwen3-Coder-30B-A3

In [11]:


# -----------------------------
# Metrics (ordinal-friendly)
# -----------------------------
def quadratic_weighted_kappa(a, b, min_rating=1, max_rating=5):
    a = np.asarray(a, dtype=int)
    b = np.asarray(b, dtype=int)

    n = max_rating - min_rating + 1
    O = np.zeros((n, n), dtype=float)
    for ra, rb in zip(a, b):
        O[ra - min_rating, rb - min_rating] += 1.0

    hist_a = O.sum(axis=1)
    hist_b = O.sum(axis=0)
    E = np.outer(hist_a, hist_b) / O.sum()

    W = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(n):
            W[i, j] = ((i - j) ** 2) / ((n - 1) ** 2)

    num = (W * O).sum()
    den = (W * E).sum()
    return 1.0 - (num / den if den != 0 else 0.0)

def spearman_rho(a, b):
    a = pd.Series(a).rank(method="average").to_numpy()
    b = pd.Series(b).rank(method="average").to_numpy()
    a = a - a.mean()
    b = b - b.mean()
    denom = (np.sqrt((a * a).sum()) * np.sqrt((b * b).sum()))
    return float((a * b).sum() / denom) if denom != 0 else np.nan

def agreement_stats(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    return {
        "exact_agreement": float(np.mean(a == b)),
        "within1_agreement": float(np.mean(np.abs(a - b) <= 1)),
        "mae": float(np.mean(np.abs(a - b))),
    }

# -----------------------------
# Column parsing + reporting
# -----------------------------
COL_RE = re.compile(r"^(wot|wtp|wtnp)_(.+)_(readability|substance_density|relevance|expert_suitability|coverage)$")

def parse_judge_columns(df: pd.DataFrame):
    """
    Parses columns like:
      wtnp_openai/gpt-oss-120b_relevance
      wtnp_Qwen/Qwen3-Coder-30B-A3B-Instruct_relevance
    into a mapping:
      mapping[prefix][criterion][judge_name] = column_name
    """
    mapping = {}
    for col in df.columns:
        m = COL_RE.match(col)
        if not m:
            continue
        prefix, judge, criterion = m.group(1), m.group(2), m.group(3)
        mapping.setdefault(prefix, {}).setdefault(criterion, {})[judge] = col
    return mapping

def compute_alignment(df: pd.DataFrame, judge1: str, judge2: str,
                      prefixes=("wtnp", "wtp", "wot"),
                      criteria=("readability","substance_density","relevance","expert_suitability","coverage"),
                      min_rating=1, max_rating=5):
    """
    Produces:
      - per (prefix, criterion) alignment between judge1 and judge2
      - per prefix (aggregated over 5 criteria)
      - overall (aggregated over prefixes+criteria)
    """
    mapping = parse_judge_columns(df)

    rows = []

    # per (prefix, criterion)
    for p in prefixes:
        for c in criteria:
            colmap = mapping.get(p, {}).get(c, {})
            if judge1 not in colmap or judge2 not in colmap:
                # skip missing pairs silently
                continue

            a = df[colmap[judge1]].astype(int).to_numpy()
            b = df[colmap[judge2]].astype(int).to_numpy()

            row = {
                "level": "prefix+criterion",
                "prefix": p,
                "criterion": c,
                "n": int(len(a)),
                "qwk": quadratic_weighted_kappa(a, b, min_rating, max_rating),
                "spearman": spearman_rho(a, b),
                **agreement_stats(a, b),
            }
            rows.append(row)

    detail = pd.DataFrame(rows)

    # per prefix (stack criteria within prefix)
    prefix_rows = []
    for p in prefixes:
        sub = detail[detail["prefix"] == p]
        if sub.empty:
            continue

        # stack all ratings across criteria for this prefix
        a_all, b_all = [], []
        for c in criteria:
            colmap = mapping.get(p, {}).get(c, {})
            if judge1 in colmap and judge2 in colmap:
                a_all.append(df[colmap[judge1]].astype(int).to_numpy())
                b_all.append(df[colmap[judge2]].astype(int).to_numpy())

        if not a_all:
            continue

        a_all = np.concatenate(a_all)
        b_all = np.concatenate(b_all)

        prefix_rows.append({
            "level": "prefix",
            "prefix": p,
            "criterion": "ALL_CRITERIA",
            "n": int(len(a_all)),
            "qwk": quadratic_weighted_kappa(a_all, b_all, min_rating, max_rating),
            "spearman": spearman_rho(a_all, b_all),
            **agreement_stats(a_all, b_all),
        })

    prefix_summary = pd.DataFrame(prefix_rows)

    # overall (stack everything)
    a_all, b_all = [], []
    for p in prefixes:
        for c in criteria:
            colmap = mapping.get(p, {}).get(c, {})
            if judge1 in colmap and judge2 in colmap:
                a_all.append(df[colmap[judge1]].astype(int).to_numpy())
                b_all.append(df[colmap[judge2]].astype(int).to_numpy())

    overall = pd.DataFrame()
    if a_all:
        a_all = np.concatenate(a_all)
        b_all = np.concatenate(b_all)
        overall = pd.DataFrame([{
            "level": "overall",
            "prefix": "ALL_PREFIXES",
            "criterion": "ALL_CRITERIA",
            "n": int(len(a_all)),
            "qwk": quadratic_weighted_kappa(a_all, b_all, min_rating, max_rating),
            "spearman": spearman_rho(a_all, b_all),
            **agreement_stats(a_all, b_all),
        }])

    return detail, prefix_summary, overall

# -----------------------------
# Example run
# -----------------------------
if __name__ == "__main__":
    # 1) Load your CSV
    csv_path = "/home/infai/Desktop/UI_SysBio_Dec03_2025/UI_SYSBIO_Sol_Lib/RUN_DEC_16/llm_judge_report_evaluation/outputs/my_dataset/DPP_corrected_few_shot_3/report_n_references/files/llm_evaluation_without_averages.csv"  # <-- change
    df = pd.read_csv(csv_path)

    # 2) Pick the judge identifiers EXACTLY as they appear in the column names
    judge_gptoss = "openai/gpt-oss-120b"
    judge_qwen   = "Qwen/Qwen3-Coder-30B-A3B-Instruct"

    detail, prefix_summary, overall = compute_alignment(
        df,
        judge1=judge_gptoss,
        judge2=judge_qwen,
        prefixes=("wtnp","wtp"),
        criteria=("readability","substance_density","relevance","expert_suitability","coverage"),
        min_rating=1,
        max_rating=5,
    )

    # 3) Print nicely
    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", 50)

    print("\n=== Per (wtnp/wtp/wot) × criterion ===")
    print(detail.sort_values(["prefix","criterion"]).to_string(index=False))

    print("\n=== Grouped by prefix (stacked over criteria) ===")
    print(prefix_summary.sort_values(["prefix"]).to_string(index=False))

    print("\n=== Overall (stacked over all prefixes+criteria) ===")
    print(overall.to_string(index=False))

    # 4) (Optional) Save outputs
    detail.to_csv("alignment_detail.csv", index=False)
    prefix_summary.to_csv("alignment_by_prefix.csv", index=False)
    overall.to_csv("alignment_overall.csv", index=False)



=== Per (wtnp/wtp/wot) × criterion ===
           level prefix          criterion   n      qwk  spearman  exact_agreement  within1_agreement      mae
prefix+criterion   wtnp           coverage 204 0.569887  0.639084         0.529412           0.931373 0.544118
prefix+criterion   wtnp expert_suitability 204 0.187320  0.401543         0.269608           0.862745 0.867647
prefix+criterion   wtnp        readability 204 0.404213  0.479841         0.627451           0.970588 0.401961
prefix+criterion   wtnp          relevance 204 0.500874  0.557459         0.539216           0.970588 0.490196
prefix+criterion   wtnp  substance_density 204 0.543575  0.545373         0.583333           0.970588 0.446078
prefix+criterion    wtp           coverage 204 0.277796  0.350638         0.401961           0.901961 0.705882
prefix+criterion    wtp expert_suitability 204 0.148918  0.377717         0.279412           0.833333 0.887255
prefix+criterion    wtp        readability 204 0.254296  0.285512       